# 03 - EDA Visualizations
**Project:** Social Media Advertising Analytics  
**Goal:** Visual exploration of channel, goal, audience, timing, and efficiency.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (12, 5)})

PALETTE = {
    'Instagram': '#FF6B35',
    'YouTube':   '#4ECDC4',
    'Facebook':  '#A855F7',
    'Twitter':   '#F72585',
    'Pinterest': '#4361EE'
}

df = pd.read_csv('../data/social_media_ads_clean.csv', parse_dates=['Date'])
print(f'Loaded: {df.shape}')

## 1. Numeric feature distributions

In [ ]:
num_cols = ['ROI', 'Conversion_Rate', 'Acquisition_Cost',
            'Clicks', 'Impressions', 'Engagement_Score',
            'Duration', 'CTR', 'Cost_per_Click', 'ROI_per_Day']

fig, axes = plt.subplots(2, 5, figsize=(22, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[col].dropna(), bins=30, color='#4361EE',
                 edgecolor='white', alpha=0.85)
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_ylabel('Count')

plt.suptitle('Numeric Feature Distributions', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../outputs/distributions.png', bbox_inches='tight')
plt.show()

## 2. Channel performance - mean ROI ranked

In [ ]:
ch_perf = (
    df.groupby('Channel_Used')
      .agg(
          Avg_ROI=('ROI', 'mean'),
          Avg_Conv=('Conversion_Rate', 'mean'),
          Avg_Cost=('Acquisition_Cost', 'mean'),
          Avg_CTR=('CTR', 'mean'),
          Count=('Campaign_ID', 'count')
      )
      .sort_values('Avg_ROI', ascending=False)
      .round(4)
)
print(ch_perf)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(
    ch_perf.index, ch_perf['Avg_ROI'],
    color=[PALETTE.get(c, '#888') for c in ch_perf.index],
    edgecolor='white', height=0.6
)
for bar, val in zip(bars, ch_perf['Avg_ROI']):
    ax.text(val + 0.02, bar.get_y() + bar.get_height() / 2,
            f'{val:.2f}x', va='center', fontsize=11, fontweight='bold')

ax.axvline(df['ROI'].mean(), color='gray', linestyle='--', alpha=0.6, label='Dataset avg')
ax.set_xlabel('Mean ROI', fontsize=12)
ax.set_title('Mean ROI by Channel', fontsize=14, fontweight='bold')
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../outputs/channel_roi.png', bbox_inches='tight')
plt.show()

## 3. Channel x Goal heatmap

In [ ]:
pivot = df.pivot_table(
    values='ROI',
    index='Campaign_Goal',
    columns='Channel_Used',
    aggfunc='mean'
).round(2)

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlGn',
            linewidths=0.5, linecolor='white',
            cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title('Mean ROI - Channel x Campaign Goal', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/channel_goal_heatmap.png', bbox_inches='tight')
plt.show()

## 4. Efficiency quadrant - Acquisition Cost vs ROI

In [ ]:
med_cost = df['Acquisition_Cost'].median()
med_roi  = df['ROI'].median()

fig, ax = plt.subplots(figsize=(12, 7))
for ch, grp in df.groupby('Channel_Used'):
    ax.scatter(
        grp['Acquisition_Cost'], grp['ROI'],
        c=PALETTE.get(ch, '#888'),
        s=grp['Engagement_Score'] * 8,
        alpha=0.55, label=ch,
        edgecolors='white', linewidth=0.3
    )

ax.axvline(med_cost, color='gray', linestyle='--', alpha=0.6)
ax.axhline(med_roi,  color='gray', linestyle='--', alpha=0.6)
ax.set_xlabel('Acquisition Cost ($)', fontsize=12)
ax.set_ylabel('ROI', fontsize=12)
ax.set_title('Efficiency Quadrant - Acquisition Cost vs ROI\n(bubble size = Engagement Score)', fontsize=13, fontweight='bold')
ax.legend(title='Channel', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.savefig('../outputs/efficiency_quadrant.png', bbox_inches='tight')
plt.show()

## 5. Monthly ROI trend

In [ ]:
monthly = df.groupby('month').agg(
    Avg_ROI=('ROI', 'mean'),
    Avg_Conv=('Conversion_Rate', 'mean')
).reset_index()

month_labels = ['Jan','Feb','Mar','Apr','May','Jun',
                'Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax1 = plt.subplots(figsize=(13, 5))
ax2 = ax1.twinx()

ax1.plot(monthly['month'], monthly['Avg_ROI'],
         color='#FF6B35', linewidth=2.5, marker='o', markersize=5, label='ROI')
ax1.fill_between(monthly['month'], monthly['Avg_ROI'], alpha=0.08, color='#FF6B35')

max_i = monthly['Avg_ROI'].idxmax()
min_i = monthly['Avg_ROI'].idxmin()
ax1.annotate(f"Peak: {monthly.loc[max_i,'Avg_ROI']:.2f}x",
             xy=(monthly.loc[max_i,'month'], monthly.loc[max_i,'Avg_ROI']),
             xytext=(0, 12), textcoords='offset points',
             ha='center', fontsize=9, color='#FF6B35', fontweight='bold')
ax1.annotate(f"Low: {monthly.loc[min_i,'Avg_ROI']:.2f}x",
             xy=(monthly.loc[min_i,'month'], monthly.loc[min_i,'Avg_ROI']),
             xytext=(0, -16), textcoords='offset points',
             ha='center', fontsize=9, color='#888')

ax2.plot(monthly['month'], monthly['Avg_Conv'] * 100,
         color='#4ECDC4', linewidth=2, linestyle='--', marker='s', markersize=4, label='Conv %')

ax1.set_xticks(range(1, 13))
ax1.set_xticklabels(month_labels)
ax1.set_ylabel('Mean ROI', color='#FF6B35', fontsize=11)
ax2.set_ylabel('Conv. Rate (%)', color='#4ECDC4', fontsize=11)
ax1.set_title('Monthly Trend - ROI and Conversion Rate', fontsize=14, fontweight='bold')
lines1, lbl1 = ax1.get_legend_handles_labels()
lines2, lbl2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, lbl1 + lbl2, loc='upper left')
plt.tight_layout()
plt.savefig('../outputs/monthly_trend.png', bbox_inches='tight')
plt.show()

## 6. Customer segment conversion rate

In [ ]:
seg = (df.groupby('Customer_Segment')['Conversion_Rate']
         .mean()
         .sort_values(ascending=False))

seg_colors = ['#FF6B35', '#4ECDC4', '#A855F7', '#4361EE']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(seg.index, seg.values * 100, color=seg_colors, edgecolor='white', width=0.6)
for bar, val in zip(bars, seg.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
            f'{val*100:.1f}%', ha='center', fontsize=11, fontweight='bold')

ax.set_ylabel('Mean Conversion Rate (%)', fontsize=12)
ax.set_title('Conversion Rate by Customer Segment', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/segment_conversion.png', bbox_inches='tight')
plt.show()

## 7. Engagement by day of week

In [ ]:
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_eng = (
    df.groupby('day_name')['Engagement_Score']
      .mean()
      .reindex(day_order)
)
day_colors = ['#4361EE'] * 5 + ['#888780'] * 2

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(day_eng.index, day_eng.values, color=day_colors, edgecolor='white', width=0.65)
for i, val in enumerate(day_eng.values):
    ax.text(i, val + 0.05, f'{val:.2f}', ha='center', fontsize=10)
ax.set_ylabel('Mean Engagement Score', fontsize=12)
ax.set_title('Engagement Score by Day of Week  (grey = weekend)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/day_engagement.png', bbox_inches='tight')
plt.show()

## 8. Correlation matrix

In [ ]:
corr_cols = ['ROI', 'Conversion_Rate', 'Acquisition_Cost',
             'Clicks', 'Impressions', 'Engagement_Score',
             'Duration', 'CTR', 'Cost_per_Click']

corr = df[corr_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, square=True,
            linewidths=0.5, cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title('Correlation Matrix - Numeric Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/correlation_matrix.png', bbox_inches='tight')
plt.show()

---
**Next:** `04_statistical_analysis.ipynb`